In [88]:
import cv2
import numpy as np

In [89]:
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

In [90]:
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

In [113]:
img = cv2.imread("images/img2.jpg")   # Replace with your image
height, width = img.shape[:2]

In [114]:
blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)
net.setInput(blob)
outputs = net.forward(output_layers)

In [115]:
boxes = []
confidences = []
class_ids = []

In [116]:
for output in outputs:
    for detection in output:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        
        if confidence > 0.3:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

In [117]:
indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.3, 0.4)

In [118]:
if len(indexes) > 0:
    indexes = np.array(indexes).reshape(-1)

    for i in indexes:
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        confidence = str(round(confidences[i], 2))
    
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img, label + " " + confidence, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [119]:
cv2.namedWindow("YOLO Detection", cv2.WINDOW_NORMAL)
cv2.imshow("YOLO Detection", img)
cv2.waitKey(0)
cv2.destroyAllWindows()